# SYP PARTS9 → Drive raw

Writes via local TEMP → copy onto Drive → `os.replace` so large
`sidet` / `icmas` files actually update under Google Drive File Stream.
(Plain in-place `to_csv` often leaves those two stale.)


In [ ]:
from sqlalchemy import create_engine
import pandas as pd

server = "KSS-PC"
database = "PARTS9"

connection_string = (
    "mssql+pyodbc://@"
    f"{server}/"
    f"{database}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&trusted_connection=yes"
)

engine = create_engine(connection_string)


In [ ]:
def read_last_5y(table):
    query = f"""
    SELECT *
    FROM dbo.{table}
    WHERE BILLDATE >= DATEADD(YEAR, -5, (SELECT MAX(BILLDATE) FROM dbo.{table}))
    """
    return pd.read_sql(query, engine)

df_sidet = read_last_5y("SIDET")
df_pidet = read_last_5y("PIDET")
df_simas = read_last_5y("SIMAS")
df_pimas = read_last_5y("PIMAS")


In [ ]:
df_icmas = pd.read_sql(
    "SELECT * FROM dbo.ICMAS",
    engine,
)


In [ ]:
import os
import shutil
import tempfile
from pathlib import Path

raw_dir = Path(r"G:\Shared drives\KCW-Data\kcw_analytics\01_raw")
os.chdir(raw_dir)
print("cwd=", os.getcwd())


def save_drive_csv(df, filename: str) -> None:
    """
    DriveFS-safe overwrite for large files (sidet/icmas).

    Plain df.to_csv(filename) often leaves the cloud object stale on Shared
    Drives while smaller files update. Write fully on local TEMP first, copy
    beside the target on G:, then os.replace (no fsync — that causes Errno 9).
    """
    path = raw_dir / filename
    fd, local_name = tempfile.mkstemp(prefix=path.stem + "_", suffix=".csv")
    os.close(fd)
    local_tmp = Path(local_name)
    drive_tmp = path.with_name(f"{path.stem}.{os.getpid()}.uploading.csv")
    try:
        df.to_csv(local_tmp, index=False, encoding="utf-8-sig")
        if drive_tmp.exists():
            drive_tmp.unlink()
        shutil.copyfile(local_tmp, drive_tmp)
        try:
            os.replace(drive_tmp, path)
        except OSError:
            if path.exists():
                path.unlink()
            os.replace(drive_tmp, path)
    finally:
        for p in (local_tmp, drive_tmp):
            if p.exists():
                try:
                    p.unlink()
                except OSError:
                    pass
    st = path.stat()
    print(
        f"wrote {filename} rows={len(df):,} size={st.st_size:,} "
        f"mtime={st.st_mtime}"
    )


# Same order as the original notebook.
save_drive_csv(df_pidet, "raw_syp_pidet_purchase_lines.csv")
save_drive_csv(df_pimas, "raw_syp_pimas_purchase_bills.csv")
save_drive_csv(df_sidet, "raw_syp_sidet_sales_lines.csv")
save_drive_csv(df_simas, "raw_syp_simas_sales_bills.csv")
save_drive_csv(df_icmas, "raw_syp_icmas_products.csv")

print("DONE")
